# Safehouse Outcome Drivers — Master Pipeline
### Pag-asa Programme Analytics | CRISP-DM End-to-End

**Purpose:** Self-contained, stakeholder-ready notebook that reproduces every pipeline
step from raw CSVs to final evaluation in a single run.

**When to use this notebook:**
- Monthly refresh after new metrics are entered
- Quick re-run to check if flagged safehouses have changed
- Onboarding: demonstrates the full pipeline logic in one place

**What it does:**
1. Regenerates the modeling-ready panel from source CSVs (Phases 1–3 logic)
2. Fits both PanelOLS models — health and education (Phase 4)
3. Produces the coefficient table and R² summary (Phase 4)
4. Flags high-volatility safehouses and saves outputs (Phase 5)
5. Prints the executive summary

**Self-healing:** If `panel_model_ready.csv` is missing or stale, this notebook
rebuilds it automatically from source CSVs before fitting the models.

---
**Estimator:** linearmodels PanelOLS — safehouse entity fixed effects, HC1-robust SEs
**R² framing:** Within-R² (directional; see Section 4 for interpretation)
**Effective N:** 244 rows (9 first-obs dropped for lag-1 features), 11 predictors

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import (
    DATA_PROCESSED, ARTIFACTS_RUNS, REPORTS,
    PANEL_READY_FILE, SCALER_STATS_FILE, COEF_TABLE_FILE,
    OUTCOME_HEALTH, OUTCOME_EDUCATION,
)
from src.data_io import load_panel
from src.features import build_model_matrix
from src.modeling import (
    build_panel_index, run_panel_ols, extract_coef_table,
    compute_residuals, save_coef_csv, MODEL_FEATURES_STD,
)
from src.evaluation import (
    load_fitted_models, flag_safehouses_report,
    save_flagged_csv, plot_residual_time_series,
)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ARTIFACTS_RUNS.mkdir(parents=True, exist_ok=True)
(REPORTS / "tables").mkdir(parents=True, exist_ok=True)
print("Setup complete.")

---
## 1. Business Understanding (summary)

**Programme question:** Which operational inputs are most associated with resident
health and education outcomes across Pag-asa safehouses?

**Models:**
| | Model A | Model B |
|--|--|--|
| Outcome | `avg_health_score` | `avg_education_progress` |
| Type | Explanatory panel OLS | Explanatory panel OLS |
| Success threshold | Within-R² ≥ 0.50 | Within-R² ≥ 0.50 |

**Framing (Phase 5 reframe):** The threshold was set pre-analysis. Both models
fall below 0.50 — a finding in itself. Unobserved factors (staffing quality,
trauma history, external events) dominate month-to-month variation. The model
is a **monitoring and directional-signal tool**, not a prediction engine.

**Stakeholders:** Pag-asa programme team, safehouse managers.

---
## 2. Data Understanding (summary)

**Sources:** `safehouse_monthly_metrics.csv`, `safehouses.csv`, `residents.csv`

| | |
|--|--|
| Raw metrics rows | 450 |
| NULL-outcome rows dropped | 197 (startup gaps, reporting lags, future scaffold) |
| Rows with lag-1 features | 244 (9 first-obs per safehouse dropped) |
| Safehouses | 9 |
| Outcome variance | Health: 50% between-SH / 50% within-SH; Education: 12% between / 88% within |

---
## 3. Data Preparation — Build Modeling Panel

Regenerates `panel_model_ready.csv` from source CSVs if missing.

In [ ]:
panel_path = DATA_PROCESSED / PANEL_READY_FILE

# Always regenerate to pick up any new monthly rows
print("Regenerating panel from source CSVs...")
panel_raw = load_panel(verbose=True)
df, _, scaler_stats = build_model_matrix(panel_raw)
df.to_csv(panel_path, index=False)
scaler_stats.to_csv(DATA_PROCESSED / SCALER_STATS_FILE, index=False)

print(f"Panel saved: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Active predictors ({len(MODEL_FEATURES_STD)}): {MODEL_FEATURES_STD}")

---
## 4. Modeling — PanelOLS with Entity Fixed Effects

Fits both models. Entity effects absorb all time-invariant safehouse-level
variation (including region). Coefficients represent within-safehouse
associations — the partial effect of a 1 SD increase in each input on outcomes,
holding the safehouse's own baseline constant.

In [ ]:
panel_indexed = build_panel_index(df)

res_a = run_panel_ols(panel_indexed, OUTCOME_HEALTH)
res_b = run_panel_ols(panel_indexed, OUTCOME_EDUCATION)

r2_a = res_a.rsquared
r2_b = res_b.rsquared

print(f"Model A  within-R2 = {r2_a:.4f}  ({OUTCOME_HEALTH})")
print(f"Model B  within-R2 = {r2_b:.4f}  ({OUTCOME_EDUCATION})")
print()
print("Interpretation:")
print(f"  Observed programme inputs explain {r2_a:.1%} of month-to-month health")
print(f"  variation and {r2_b:.1%} of education variation within each safehouse.")
print(f"  The remainder reflects unobserved factors (staffing, trauma history, context).")
print(f"  Coefficients below should be read as directional signals, not causal estimates.")

---
## 4.1. Coefficient Table (both models, side-by-side)

**β** = change in outcome per 1 SD increase in the predictor (within-safehouse)
**Significance:** `.` p<0.10 · `*` p<0.05 · `**` p<0.01 · `***` p<0.001

In [ ]:
coef_table = extract_coef_table(res_a, res_b)

display_tbl = coef_table.copy()
display_tbl.index = [f.replace("_std", "") for f in display_tbl.index]
display_tbl.columns.name = None
pd.set_option("display.float_format", "{:.4f}".format)
display(display_tbl)

# Save
coef_path = ARTIFACTS_RUNS / COEF_TABLE_FILE
save_coef_csv(coef_table, coef_path)

---
## 4.2. Coefficient Plot — Significant Predictors Only

In [ ]:
sig_mask_a = coef_table["p_A"] < 0.10
sig_mask_b = coef_table["p_B"] < 0.10
any_sig = sig_mask_a | sig_mask_b

plot_df = coef_table[any_sig].copy()
plot_df.index = [f.replace("_std", "") for f in plot_df.index]

if len(plot_df) == 0:
    print("No predictors significant at p<0.10 in either model.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(plot_df) * 0.6 + 1)), sharey=True)

    for ax, beta_col, se_col, p_col, label, color in [
        (axes[0], "beta_A", "se_A", "p_A", f"Model A\n{OUTCOME_HEALTH}", "steelblue"),
        (axes[1], "beta_B", "se_B", "p_B", f"Model B\n{OUTCOME_EDUCATION}", "seagreen"),
    ]:
        sub = coef_table[coef_table[p_col] < 0.10].copy()
        sub.index = [f.replace("_std", "") for f in sub.index]
        if sub.empty:
            ax.text(0.5, 0.5, "No predictors\nsignificant at p<0.10",
                    ha="center", va="center", transform=ax.transAxes, fontsize=9)
            ax.set_title(label, fontsize=10)
            continue
        sub = sub.sort_values(beta_col)
        ax.barh(sub.index, sub[beta_col], xerr=1.96 * sub[se_col],
                color=color, alpha=0.8, edgecolor="white", capsize=3)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
        ax.set_title(label, fontsize=10)
        ax.set_xlabel("beta (SD units, 95% CI)")

    plt.suptitle("Significant Predictors (p<0.10) — Within-Safehouse Associations",
                 fontsize=12)
    plt.tight_layout()
    plt.show()

---
## 5. Evaluation — Flagged Safehouses & Residual Diagnostics

Safehouses are flagged when their **residual variance** exceeds mean + 2 SD
across all safehouses — meaning their outcomes swing more than the observed
programme inputs can account for. These warrant a case review.

In [ ]:
resid_by_sh    = compute_residuals(res_a, res_b)
flagged_report = flag_safehouses_report(resid_by_sh, df)

flagged_health = flagged_report[flagged_report["var_A_flag"]]["safehouse_id"].tolist()
flagged_edu    = flagged_report[flagged_report["var_B_flag"]]["safehouse_id"].tolist()

print(f"Flagged for health volatility    : {flagged_health if flagged_health else 'None'}")
print(f"Flagged for education volatility : {flagged_edu if flagged_edu else 'None'}")
print()
display(flagged_report)

# Save
flagged_path = save_flagged_csv(flagged_report, REPORTS)
print(f"\nFlagged report saved -> {flagged_path}")

---
## 5.1. Residual Variance by Safehouse

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, flag_col, label in [
    (axes[0], "var_A", "var_A_flag", f"Model A - {OUTCOME_HEALTH}"),
    (axes[1], "var_B", "var_B_flag", f"Model B - {OUTCOME_EDUCATION}"),
]:
    threshold = resid_by_sh[col].mean() + 2 * resid_by_sh[col].std()
    colors = resid_by_sh[flag_col].map({True: "crimson", False: "steelblue"})

    ax.bar(resid_by_sh.index.astype(str), resid_by_sh[col],
           color=colors, edgecolor="white")
    ax.axhline(threshold, color="crimson", linewidth=0.9, linestyle="--", alpha=0.7,
               label=f"mean + 2 SD = {threshold:.4f}")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Safehouse ID")
    ax.set_ylabel("Residual variance")
    ax.legend(fontsize=8)

    flagged_ids = resid_by_sh[resid_by_sh[flag_col]].index.tolist()
    if flagged_ids:
        ax.annotate(f"Flagged: SH{flagged_ids}", xy=(0.02, 0.95),
                    xycoords="axes fraction", fontsize=8, color="crimson", va="top")

normal_patch = mpatches.Patch(color="steelblue", label="Within threshold")
flag_patch   = mpatches.Patch(color="crimson",   label="Above mean + 2 SD")
fig.legend(handles=[normal_patch, flag_patch], loc="lower center",
           ncol=2, bbox_to_anchor=(0.5, -0.05), fontsize=9)
plt.suptitle("Per-Safehouse Residual Variance (variance-based flagging)", fontsize=12)
plt.tight_layout()
plt.show()

---
## 5.2. Residuals Over Time — Flagged Safehouses Highlighted

In [ ]:
fig = plot_residual_time_series(res_a, res_b, panel_indexed,
                                flagged_health, flagged_edu)
plt.show()

---
## 6. Executive Summary

> **What we found, in plain language — for programme managers.**

In [ ]:
exec_summary_path = ROOT / "reports" / "executive_summary.md"
if exec_summary_path.exists():
    from IPython.display import Markdown, display as ipy_display
    ipy_display(Markdown(exec_summary_path.read_text(encoding="utf-8")))
else:
    print(f"Executive summary not found at {exec_summary_path}")
    print("Run the Phase 5 notebook to generate it.")

---
## 7. Pipeline Metadata

Captured at run time for reproducibility.

In [ ]:
from datetime import date
import linearmodels
import pandas as pd
import numpy as np

metadata = pd.DataFrame([
    {"field": "run_date",           "value": date.today().isoformat()},
    {"field": "panel_rows",         "value": len(df)},
    {"field": "panel_columns",      "value": len(df.columns)},
    {"field": "n_predictors",       "value": len(MODEL_FEATURES_STD)},
    {"field": "safehouses",         "value": df["safehouse_id"].nunique()},
    {"field": "health_within_r2",   "value": round(res_a.rsquared, 4)},
    {"field": "edu_within_r2",      "value": round(res_b.rsquared, 4)},
    {"field": "flagged_health",      "value": str(flagged_health)},
    {"field": "flagged_edu",         "value": str(flagged_edu)},
    {"field": "estimator",          "value": "linearmodels.PanelOLS entity effects HC1-robust"},
    {"field": "linearmodels_ver",   "value": linearmodels.__version__},
    {"field": "pandas_ver",         "value": pd.__version__},
    {"field": "numpy_ver",          "value": np.__version__},
    {"field": "panel_csv",          "value": str(panel_path)},
    {"field": "coef_csv",           "value": str(coef_path)},
    {"field": "flagged_csv",        "value": str(flagged_path)},
])
display(metadata)